# Ноутбук 01 — Предобработка данных и конструирование признаков
**Подразделы 2.1 и 2.4 ПЗ**

Запускать после `00_eda_dataset_overview.ipynb`.

Артефакты:
- `data/interim/train_clean.parquet`
- `data/interim/weekly_sales.parquet`
- `data/processed/features_train.parquet`
- `data/processed/features_test.parquet`
- `reports/tables/table_2_1_missing_values.csv`
- `reports/tables/table_2_4_feature_space.csv`

## Ячейка 0 — Импорты и конфигурация

In [1]:
import sys
sys.path.insert(0, "..")
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from src.config import (
    DATA_INT, DATA_PROC, TABLES,
    DATE_COL, STORE_COL, FAMILY_COL, GROUP_KEYS,
    TARGET, TRAIN_CUTOFF,
    LAG_WEEKS, ROLLING_WINDOWS, PROMOTION_LAG,
    OIL_PRICE_MIN, OIL_PRICE_MAX, SALES_MIN, SALES_MAX,
)
from src.io.preprocess import (
    load_raw_files,
    fill_oil_gaps,
    remove_duplicates,
    save_interim,
)

DATA_INT.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

print("Конфигурация загружена.")
print(f"  TRAIN_CUTOFF   : {TRAIN_CUTOFF}")
print(f"  LAG_WEEKS      : {LAG_WEEKS}")
print(f"  ROLLING_WINDOWS: {ROLLING_WINDOWS}")

Конфигурация загружена.
  TRAIN_CUTOFF   : 2017-05-01
  LAG_WEEKS      : [1, 2, 4, 9, 12, 52]
  ROLLING_WINDOWS: [4, 12]


## Ячейка 1 — Загрузка сырых данных

In [2]:
data = load_raw_files()

train        = data["train"]
stores       = data["stores"]
oil          = data["oil"]
holidays     = data["holidays"]
transactions = data["transactions"]

# ── Валидация схемы входных файлов ────────────────────────────────────────
_REQUIRED_COLS = {
    "train"       : {"date", "store_nbr", "family", "sales", "onpromotion"},
    "stores"      : {"store_nbr", "type", "cluster"},
    "oil"         : {"date", "dcoilwtico"},
    "holidays"    : {"date", "type", "locale", "transferred"},
    "transactions": {"date", "store_nbr", "transactions"},
}
for name, req in _REQUIRED_COLS.items():
    actual = set(data[name].columns)
    missing = req - actual
    assert not missing, f"Файл {name!r}: отсутствуют столбцы {missing}"

# ── Валидация дат ─────────────────────────────────────────────────────────
train[DATE_COL] = pd.to_datetime(train[DATE_COL])
assert train[DATE_COL].notna().all(), "train: обнаружены NaT в столбце date"
assert train[DATE_COL].min() >= pd.Timestamp("2013-01-01"), "train: дата ниже ожидаемого диапазона"
assert train[DATE_COL].max() <= pd.Timestamp("2017-12-31"), "train: дата выше ожидаемого диапазона"

# ── Валидация sales ───────────────────────────────────────────────────────
assert train["sales"].dtype in [float, int, "float64", "int64"], "train: sales не числовой тип"

print("Загружено строк:")
for name, df_ in data.items():
    print(f"  {name:15s}: {len(df_):>10,} строк  |  {df_.shape[1]} столбцов")
print("\nВалидация схемы: ОК")

Загружено строк:
  train          :  3,000,888 строк  |  6 столбцов
  test           :     28,512 строк  |  5 столбцов
  stores         :         54 строк  |  5 столбцов
  oil            :      1,218 строк  |  2 столбцов
  holidays       :        350 строк  |  6 столбцов
  transactions   :     83,488 строк  |  3 столбцов

Валидация схемы: ОК


## Ячейка 2 — Заполнение пропусков в ценах на нефть

In [3]:
n_missing_before = oil["dcoilwtico"].isna().sum()
oil_clean = fill_oil_gaps(oil)
n_missing_after = oil_clean["dcoilwtico"].isna().sum()

assert n_missing_after == 0, f"После fill_oil_gaps остались {n_missing_after} пропусков"

oil_actual_min = oil_clean["dcoilwtico"].min()
oil_actual_max = oil_clean["dcoilwtico"].max()
assert oil_actual_min >= 10.0, f"dcoilwtico: неправдоподобно малое значение {oil_actual_min:.1f}"
assert oil_actual_max <= 200.0, f"dcoilwtico: неправдоподобно большое значение {oil_actual_max:.1f}"

print(f"Пропуски в dcoilwtico до очистки : {n_missing_before}")
print(f"Пропуски в dcoilwtico после очистки: {n_missing_after}  — ОК")
print(f"Диапазон dcoilwtico: {oil_actual_min:.1f} – {oil_actual_max:.1f} (ожидается ≈26–111 по п. 2.2.1 ПЗ)")

Пропуски в dcoilwtico до очистки : 43
Пропуски в dcoilwtico после очистки: 0  — ОК
Диапазон dcoilwtico: 26.2 – 110.6 (ожидается ≈26–111 по п. 2.2.1 ПЗ)


## Ячейка 3 — Удаление дублей и отрицательных продаж

In [4]:
train = remove_duplicates(train)

n_negative = (train["sales"] < 0).sum()
train["sales"] = train["sales"].clip(lower=0)

assert (train["sales"] < 0).sum() == 0, "После clip(lower=0) остались отрицательные продажи"

print(f"Отрицательных значений sales (усечено до 0): {n_negative}")
print(f"Уникальных (store_nbr, family, date) после remove_duplicates: {len(train):,}")

Отрицательных значений sales (усечено до 0): 0
Уникальных (store_nbr, family, date) после remove_duplicates: 3,000,888


## Ячейка 4 — Слияние всех таблиц

In [5]:
df = train.merge(stores, on=STORE_COL, how="left")

n_store_nan = df["cluster"].isna().sum()
assert n_store_nan == 0, f"store_nbr без соответствия в stores.csv: {n_store_nan} строк"

oil_clean[DATE_COL] = pd.to_datetime(oil_clean[DATE_COL])
df = df.merge(oil_clean[[DATE_COL, "dcoilwtico"]], on=DATE_COL, how="left")

transactions[DATE_COL] = pd.to_datetime(transactions[DATE_COL])
df = df.merge(transactions, on=[DATE_COL, STORE_COL], how="left")
df["transactions"] = df["transactions"].fillna(0).astype(int)

holidays[DATE_COL] = pd.to_datetime(holidays[DATE_COL])
nat_holidays = (
    holidays[
        (holidays["locale"] == "National") &
        (holidays["transferred"] == False)
    ][[DATE_COL]]
    .drop_duplicates()
    .assign(is_holiday=1)
)
df = df.merge(nat_holidays, on=DATE_COL, how="left")
df["is_holiday"] = df["is_holiday"].fillna(0).astype(int)

assert df.shape[0] == len(train), "Число строк изменилось после left-merge — проверить ключи"
assert df["transactions"].isna().sum() == 0, "NaN в transactions после fillna"
assert df["is_holiday"].isin([0, 1]).all(), "is_holiday содержит значения кроме 0/1"

print(f"Итоговая таблица: {df.shape[0]:,} строк  |  {df.shape[1]} столбцов")
print(f"Диапазон дат: {df[DATE_COL].min().date()} — {df[DATE_COL].max().date()}")
print(f"NaN в столбце transactions: {df['transactions'].isna().sum()}  — ОК")

Итоговая таблица: 3,000,888 строк  |  13 столбцов
Диапазон дат: 2013-01-01 — 2017-08-15
NaN в столбце transactions: 0  — ОК


## Ячейка 5 — Таблица пропущенных значений (Таблица 2.1 ПЗ)

In [6]:
def missing_summary(df_: pd.DataFrame, name: str) -> pd.DataFrame:
    total = df_.isna().sum()
    pct   = (total / len(df_) * 100).round(2)
    return pd.DataFrame({
        "Таблица"   : name,
        "Столбец"   : total.index,
        "Пропусков" : total.values,
        "Доля, %"   : pct.values,
    })

missing_parts = [
    missing_summary(train,        "train"),
    missing_summary(oil_clean,    "oil"),
    missing_summary(transactions, "transactions"),
    missing_summary(df,           "merged_df"),
]
missing_df = pd.concat(missing_parts, ignore_index=True)
missing_df = missing_df[missing_df["Пропусков"] > 0]

missing_df.to_csv(TABLES / "table_2_1_missing_values.csv", index=False)
print("Таблица 2.1 сохранена.")
display(missing_df)

SEASON = 52
df = df.sort_values([STORE_COL, FAMILY_COL, DATE_COL])
df["sales_diff52w"] = (
    df.groupby(GROUP_KEYS)["sales"]
    .transform(lambda x: x - x.shift(SEASON))
    .fillna(0)
)
assert df["sales_diff52w"].isna().sum() == 0, "NaN в sales_diff52w после fillna"
print("sales_diff52w добавлен. NaN: 0  — ОК")

Таблица 2.1 сохранена.


,Таблица,Столбец,Пропусков,"Доля, %"


sales_diff52w добавлен. NaN: 0  — ОК


## Ячейка 6 — Сохранение очищенной таблицы в `data/interim/`

In [7]:
save_interim(df, "train_clean")
print(f"Сохранено: {DATA_INT / 'train_clean.parquet'}")

Сохранено: D:\user\Documents\КПАиММО\store-sales-kr\data\interim\train_clean.parquet
Сохранено: D:\user\Documents\КПАиММО\store-sales-kr\data\interim\train_clean.parquet


## Ячейка 7 — Недельная агрегация суммарных продаж

In [8]:
weekly = (
    df.groupby(DATE_COL)["sales"]
    .sum()
    .resample("W-MON")
    .sum()
    .rename("sales")
)

assert len(weekly) > 0, "Недельный ряд пуст"
assert weekly.isna().sum() == 0, "NaN в недельном ряду"

weekly_path = DATA_INT / "weekly_sales.parquet"
weekly.to_frame().to_parquet(weekly_path)

print(f"Недельный ряд: {len(weekly)} наблюдений")
print(f"  от {weekly.index[0].date()} до {weekly.index[-1].date()}")
print(f"Сохранено: {weekly_path}")

Недельный ряд: 242 наблюдений
  от 2013-01-07 до 2017-08-21
Сохранено: D:\user\Documents\КПАиММО\store-sales-kr\data\interim\weekly_sales.parquet


## Ячейка 8 — Конструирование временны́х признаков

In [9]:
CYCLE_PERIOD = 52

df["year"]        = df[DATE_COL].dt.year
df["month"]       = df[DATE_COL].dt.month
df["week"]        = df[DATE_COL].dt.isocalendar().week.astype(int)
df["day_of_week"] = df[DATE_COL].dt.dayofweek
df["day_of_year"] = df[DATE_COL].dt.dayofyear
df["week_sin"]    = np.sin(2 * np.pi * df["week"] / CYCLE_PERIOD)
df["week_cos"]    = np.cos(2 * np.pi * df["week"] / CYCLE_PERIOD)

assert df["week_sin"].between(-1.0, 1.0).all(), "week_sin выходит за [-1, 1]"
assert df["week_cos"].between(-1.0, 1.0).all(), "week_cos выходит за [-1, 1]"

print(f"Временны́е признаки добавлены. week_sin ∈ [{df['week_sin'].min():.3f}, {df['week_sin'].max():.3f}]")

Временны́е признаки добавлены. week_sin ∈ [-1.000, 1.000]


## Ячейка 9 — Лаговые и скользящие признаки

In [10]:
df = df.sort_values([STORE_COL, FAMILY_COL, DATE_COL])

for lag in LAG_WEEKS:
    df[f"sales_lag_{lag}w"] = df.groupby(GROUP_KEYS)["sales"].shift(lag)

for window in ROLLING_WINDOWS:
    df[f"sales_roll_{window}w_mean"] = (
        df.groupby(GROUP_KEYS)["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    df[f"sales_roll_{window}w_std"] = (
        df.groupby(GROUP_KEYS)["sales"]
        .transform(lambda x: x.shift(1).rolling(window, min_periods=2).std())
        .fillna(0)
    )

lag_cols = [c for c in df.columns if c.startswith("sales_lag_") or c.startswith("sales_roll_")]
std_cols  = [c for c in lag_cols if c.endswith("_std")]

assert df[std_cols].isna().sum().sum() == 0, "NaN в скользящих std после fillna(0)"

print(f"Лаговых и скользящих признаков добавлено: {len(lag_cols)}")
print(f"  {lag_cols}")

Лаговых и скользящих признаков добавлено: 10
  ['sales_lag_1w', 'sales_lag_2w', 'sales_lag_4w', 'sales_lag_9w', 'sales_lag_12w', 'sales_lag_52w', 'sales_roll_4w_mean', 'sales_roll_4w_std', 'sales_roll_12w_mean', 'sales_roll_12w_std']


## Ячейка 10 — Разбиение на обучающую и тестовую выборки

In [11]:
cutoff = pd.Timestamp(TRAIN_CUTOFF)

df_train = df[df[DATE_COL] < cutoff].copy()
df_test  = df[df[DATE_COL] >= cutoff].copy()

assert len(df_train) > 0, "Обучающая выборка пуста — проверить TRAIN_CUTOFF"
assert len(df_test) > 0,  "Тестовая выборка пуста — проверить TRAIN_CUTOFF"
assert df_train[DATE_COL].max() < cutoff, "Утечка: обучающая выборка содержит даты >= TRAIN_CUTOFF"
assert df_test[DATE_COL].min() >= cutoff, "Утечка: тестовая выборка содержит даты < TRAIN_CUTOFF"

print(f"Обучающая: {len(df_train):,} строк  ({df_train[DATE_COL].min().date()} — {df_train[DATE_COL].max().date()})")
print(f"Тестовая : {len(df_test):,} строк  ({df_test[DATE_COL].min().date()} — {df_test[DATE_COL].max().date()})")
print("Перекрытие по времени: ОТСУТСТВУЕТ — ОК")

Обучающая: 2,810,214 строк  (2013-01-01 — 2017-04-30)
Тестовая : 190,674 строк  (2017-05-01 — 2017-08-15)
Перекрытие по времени: ОТСУТСТВУЕТ — ОК


## Ячейка 11 — Сохранение финальных датасетов в `data/processed/`

In [12]:
train_path = DATA_PROC / "features_train.parquet"
test_path  = DATA_PROC / "features_test.parquet"

df_train.to_parquet(train_path, index=False)
df_test.to_parquet(test_path,  index=False)

print(f"Сохранено: {train_path}")
print(f"Сохранено: {test_path}")

Сохранено: D:\user\Documents\КПАиММО\store-sales-kr\data\processed\features_train.parquet
Сохранено: D:\user\Documents\КПАиММО\store-sales-kr\data\processed\features_test.parquet


---
# ПОДРАЗДЕЛ 2.4 — ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ

Ниже реализован полный конвейер конструирования признаков
согласно подразделу 2.4 пояснительной записки (пп. 2.4.1–2.4.7).

Конвейер применяется к `df_train` и `df_test` через `build_feature_matrix()`
из `src/features/build_features.py`. Нормализация (п. 2.4.6) вынесена
в `src/features/scaling.py` и применяется только для LSTM и упругой сети.

## Ячейка 2.4.1 — Лаговые признаки

In [13]:
from src.features.build_features import weekly_aggregation, add_lag_features
from src.config import LAG_WEEKS, PROMOTION_LAG

# ── Агрегируем ВЕСЬ df, затем разбиваем по недельным меткам ──────────────
# Разбивка в ячейке 10 была по дневным датам — после weekly_aggregation
# метка 2017-06-05 попадала в обе выборки. Вследствие этого разбивку
# переносим сюда, ПОСЛЕ агрегации.
_DAILY_COLS = [DATE_COL, STORE_COL, FAMILY_COL, "sales", "onpromotion"]
cutoff = pd.Timestamp(TRAIN_CUTOFF)

_df_all  = weekly_aggregation(df[_DAILY_COLS].copy())
df_train = _df_all[_df_all[DATE_COL] <  cutoff].copy()
df_test  = _df_all[_df_all[DATE_COL] >= cutoff].copy()

# ── Лаговые признаки ──────────────────────────────────────────────────────
df_train = add_lag_features(df_train, lag_weeks=LAG_WEEKS)
df_test  = add_lag_features(df_test,  lag_weeks=LAG_WEEKS)
# ── Винсоризация onpromotion_lag1 (fit ONLY on train) ─────────────────────
# Порог — 99-й перцентиль обучающей выборки.
# Вследствие правостороннего распределения счётчика промоакций
# тестовые точки могут превышать диапазон train — clip устраняет выброс
# без утечки данных, поскольку порог вычисляется только по df_train.
_promo_cap = df_train["onpromotion_lag1"].quantile(0.99)

df_train["onpromotion_lag1"] = df_train["onpromotion_lag1"].clip(upper=_promo_cap)
df_test["onpromotion_lag1"]  = df_test["onpromotion_lag1"].clip(upper=_promo_cap)

print(f"onpromotion_lag1: порог 99-го перцентиля (train) = {_promo_cap:.1f}")
print(f"  train max после clip: {df_train['onpromotion_lag1'].max():.1f}")
print(f"  test  max после clip: {df_test['onpromotion_lag1'].max():.1f}")

expected_lag_cols = [f"lag_{w}" for w in LAG_WEEKS] + ["onpromotion_lag1"]
for col in expected_lag_cols:
    assert col in df_train.columns, f"Столбец {col!r} отсутствует в df_train"
    assert col in df_test.columns,  f"Столбец {col!r} отсутствует в df_test"

for w in [1, 2, 4]:
    col = f"lag_{w}"
    max_lag  = df_train[col].dropna().max()
    max_sale = df_train["sales_weekly"].max()
    assert max_lag <= max_sale * 1.01, (
        f"{col}: max={max_lag:.0f} превышает max(sales_weekly)={max_sale:.0f} — возможна утечка"
    )

neg_promo = (df_train["onpromotion_lag1"].dropna() < 0).sum()
assert neg_promo == 0, f"onpromotion_lag1: обнаружено {neg_promo} отрицательных значений"

print(f"lag-признаки сформированы: {expected_lag_cols}")
print(f"NaN в lag_1  (обучающая): {df_train['lag_1'].isna().sum():>6,}  (норма: первые {LAG_WEEKS[0]} строк группы)")
print(f"NaN в lag_52 (обучающая): {df_train['lag_52'].isna().sum():>6,}  (норма: первые 52 строки группы)")
print(f"NaN в onpromotion_lag1   : {df_train['onpromotion_lag1'].isna().sum():>6,}")
print("Проверка диапазона lag_1–lag_4: ОК")
print("Проверка неотрицательности onpromotion_lag1: ОК")

onpromotion_lag1: порог 99-го перцентиля (train) = 254.0
  train max после clip: 254.0
  test  max после clip: 254.0
lag-признаки сформированы: ['lag_1', 'lag_2', 'lag_4', 'lag_9', 'lag_12', 'lag_52', 'onpromotion_lag1']
NaN в lag_1  (обучающая):  1,782  (норма: первые 1 строк группы)
NaN в lag_52 (обучающая): 92,664  (норма: первые 52 строки группы)
NaN в onpromotion_lag1   :  1,782
Проверка диапазона lag_1–lag_4: ОК
Проверка неотрицательности onpromotion_lag1: ОК


## Ячейка 2.4.2 — Скользящие статистики

In [14]:
from src.features.build_features import add_rolling_features

df_train = add_rolling_features(df_train)
df_test  = add_rolling_features(df_test)

target_roll_cols = ["rolling_mean_4", "rolling_std_4", "rolling_mean_12"]
for col in target_roll_cols:
    assert col in df_train.columns, f"Столбец {col!r} отсутствует в df_train"

std_cols_roll = [c for c in df_train.columns if c.startswith("rolling_std")]
nan_std_train = df_train[std_cols_roll].isna().sum().sum()
nan_std_test  = df_test[std_cols_roll].isna().sum().sum()
assert nan_std_train == 0, f"NaN в rolling_std (обучающая): {nan_std_train}"
assert nan_std_test  == 0, f"NaN в rolling_std (тестовая): {nan_std_test}"

for col in ["rolling_mean_4", "rolling_mean_12"]:
    neg_count = (df_train[col].dropna() < 0).sum()
    assert neg_count == 0, f"{col}: обнаружено {neg_count} отрицательных значений"

extreme_std = (df_train["rolling_std_4"] > df_train["rolling_mean_4"] * 10).sum()
if extreme_std > 0:
    print(f"[WARN] rolling_std_4 > 10 × rolling_mean_4 в {extreme_std} строках — проверить на выбросы")

for col in target_roll_cols:
    print(f"  {col}: mean={df_train[col].mean():.1f}, std={df_train[col].std():.1f}, NaN={df_train[col].isna().sum()}")
print("FIX-2: NaN в rolling_std = 0 — ОК")

  rolling_mean_4: mean=2419.2, std=7059.3, NaN=1782
  rolling_std_4: mean=291.6, std=1169.5, NaN=0
  rolling_mean_12: mean=2383.0, std=6909.8, NaN=1782
FIX-2: NaN в rolling_std = 0 — ОК


## Ячейка 2.4.3 — Циклическое кодирование календарных признаков

In [15]:
from src.features.build_features import add_calendar_features

df_train = add_calendar_features(df_train)
df_test  = add_calendar_features(df_test)

calendar_cols = ["week_of_year_sin", "week_of_year_cos", "month_sin", "month_cos"]
for col in calendar_cols:
    assert col in df_train.columns, f"Столбец {col!r} отсутствует в df_train"
    assert col in df_test.columns,  f"Столбец {col!r} отсутствует в df_test"

for col in calendar_cols:
    col_min = df_train[col].min()
    col_max = df_train[col].max()
    assert col_min >= -1.0 - 1e-9, f"{col}: min={col_min:.6f} < -1"
    assert col_max <=  1.0 + 1e-9, f"{col}: max={col_max:.6f} > +1"

week_identity  = df_train["week_of_year_sin"]**2 + df_train["week_of_year_cos"]**2
month_identity = df_train["month_sin"]**2 + df_train["month_cos"]**2
assert week_identity.sub(1.0).abs().max() < 1e-9
assert month_identity.sub(1.0).abs().max() < 1e-9

for col in calendar_cols:
    assert df_train[col].isna().sum() == 0, f"NaN в {col} (обучающая)"
    assert df_test[col].isna().sum()  == 0, f"NaN в {col} (тестовая)"

print("Циклические признаки: ОК")
print(f"  sin²+cos²=1 (week): {week_identity.mean():.10f}")
print(f"  sin²+cos²=1 (month): {month_identity.mean():.10f}")
for col in calendar_cols:
    print(f"  {col}: [{df_train[col].min():.4f}, {df_train[col].max():.4f}], NaN=0")

Циклические признаки: ОК
  sin²+cos²=1 (week): 1.0000000000
  sin²+cos²=1 (month): 1.0000000000
  week_of_year_sin: [-1.0000, 1.0000], NaN=0
  week_of_year_cos: [-1.0000, 1.0000], NaN=0
  month_sin: [-1.0000, 1.0000], NaN=0
  month_cos: [-1.0000, 1.0000], NaN=0


## Ячейка 2.4.4 — Праздничные бинарные признаки

Формируются `is_holiday`, `is_national`, `is_regional`.
Признак `is_national` (174 записи) представляет наибольший прогностический
интерес, поскольку охватывает всю сеть и создаёт рождественские пики
(STL-анализ, п. 2.2.3 ПЗ). Признак `is_regional` (24 события) сохранён,
поскольку создаёт локальные скачки, не объяснимые другими признаками.

**Примечание о пересечении is_national и is_regional:**
На недельном уровне агрегации (W-MON) пересечение `is_national=1` и
`is_regional=1` в одной строке допустимо — национальный и региональный
праздники могут попасть в разные дни одной и той же недели.
Это не ошибка данных, а артефакт агрегации (FIX-3).

In [16]:
from src.features.build_features import add_holiday_feature

df_train = add_holiday_feature(df_train, holidays)
df_test  = add_holiday_feature(df_test,  holidays)

# ── Проверка 1: все три столбца созданы ──────────────────────────────────
holiday_cols = ["is_holiday", "is_national", "is_regional"]
for col in holiday_cols:
    assert col in df_train.columns, f"{col!r} отсутствует в df_train"
    assert col in df_test.columns,  f"{col!r} отсутствует в df_test"

# ── Проверка 2: только бинарные значения 0/1 ────────────────────────────
for col in holiday_cols:
    assert df_train[col].isin([0, 1]).all(), f"{col}: найдены значения кроме 0 и 1 (train)"
    assert df_test[col].isin([0, 1]).all(),  f"{col}: найдены значения кроме 0 и 1 (test)"

# ── Проверка 3 (FIX-3): пересечение допустимо на W-MON агрегации ─────────
# Национальный и региональный праздники могут попасть в разные дни
# одной недели — пересечение is_national=1 & is_regional=1 не является
# ошибкой данных, выводим как информационное сообщение.
overlap_train = int((df_train["is_national"].astype(bool) & df_train["is_regional"].astype(bool)).sum())
overlap_test  = int((df_test["is_national"].astype(bool) & df_test["is_regional"].astype(bool)).sum())
if overlap_train > 0:
    print(f"[INFO] is_national ∩ is_regional (обучающая): {overlap_train} строк — норма для W-MON агрегации")
if overlap_test > 0:
    print(f"[INFO] is_national ∩ is_regional (тестовая) : {overlap_test} строк — норма для W-MON агрегации")

# ── Проверка 4: is_holiday >= is_national и is_regional (супермножество) ─
assert (df_train["is_holiday"] >= df_train["is_national"]).all(), "is_holiday < is_national — нарушена логика объединения"
assert (df_train["is_holiday"] >= df_train["is_regional"]).all(), "is_holiday < is_regional — нарушена логика объединения"

# ── Проверка 5: ожидаемое число национальных праздников ──────────────────
n_national_train = df_train["is_national"].sum()
n_regional_train = df_train["is_regional"].sum()
print(f"Строк с is_national=1 (обучающая): {n_national_train:,}")
print(f"Строк с is_regional=1 (обучающая): {n_regional_train:,}")
print(f"Пересечение is_national ∩ is_regional: {overlap_train} строк (норма для W-MON)")
print("is_holiday ⊇ is_national ∪ is_regional: ОК")

[INFO] add_holiday_feature: 7128 строк с is_national=1 и is_regional=1 (нац. и рег. праздники в одну неделю — допустимо на W-MON агрегации)
[INFO] is_national ∩ is_regional (обучающая): 7128 строк — норма для W-MON агрегации
Строк с is_national=1 (обучающая): 80,190
Строк с is_regional=1 (обучающая): 24,948
Пересечение is_national ∩ is_regional: 7128 строк (норма для W-MON)
is_holiday ⊇ is_national ∪ is_regional: ОК


## Ячейка 2.4.5 — Кодирование категориальных характеристик магазинов

In [ ]:
from src.features.build_features import add_store_features
import pandas as pd

# ── Кодирование характеристик магазина ───────────────────────────────────────
df_train = add_store_features(df_train, stores)
df_test  = add_store_features(df_test,  stores)

expected_store_cols = ["cluster"] + [f"store_type_{t}" for t in ["A","B","C","D","E"]]
for col in expected_store_cols:
    assert col in df_train.columns, f"{col!r} отсутствует в df_train"
    assert col in df_test.columns,  f"{col!r} отсутствует в df_test"

store_type_cols = [c for c in df_train.columns if c.startswith("store_type_")]
n_onehot = len(store_type_cols)
assert n_onehot == 5, f"Ожидалось 5 one-hot столбцов store_type, получено {n_onehot}"

row_sums = df_train[store_type_cols].sum(axis=1)
invalid_rows = (row_sums != 1).sum()
assert invalid_rows == 0, f"{invalid_rows} строк не принадлежат ровно одному типу магазина"

cluster_min = df_train["cluster"].min()
cluster_max = df_train["cluster"].max()
assert cluster_min >= 1,  f"cluster: минимум {cluster_min} < 1"
assert cluster_max <= 17, f"cluster: максимум {cluster_max} > 17"

train_type_set = set(c for c in df_train.columns if c.startswith("store_type_"))
test_type_set  = set(c for c in df_test.columns  if c.startswith("store_type_"))
assert train_type_set == test_type_set

print(f"One-hot столбцов store_type: {n_onehot}  — ОК")
print(f"store_type столбцы: {store_type_cols}")
print(f"cluster: [{cluster_min}, {cluster_max}] — ОК")
print("Каждая строка — ровно 1 тип магазина: ОК")
print("Признаковое пространство train == test: ОК")

# ── Кодирование семейств товаров (FIX-FAMILY) ─────────────────────────────────
# Без one-hot family модель не различает масштаб ряда BEVERAGES и BOOKS,
# поскольку family используется только как ключ groupby при расчёте лагов.
# Вследствие этого RF/XGBoost усредняют 1 782 гетерогенных ряда без контекста
# масштаба — это структурная причина MAPE = 68–168 %.
_KNOWN_FAMILIES = sorted(df_train[FAMILY_COL].unique())
_EXPECTED_N_FAMILIES = 33

family_dummies_train = pd.get_dummies(
    df_train[FAMILY_COL], prefix="family", dtype=int
)
family_dummies_test = pd.get_dummies(
    df_test[FAMILY_COL], prefix="family", dtype=int
)

# Выравниваем столбцы: тест может не содержать редких семейств
family_dummies_test = family_dummies_test.reindex(
    columns=family_dummies_train.columns, fill_value=0
)

df_train = pd.concat([df_train, family_dummies_train], axis=1)
df_test  = pd.concat([df_test,  family_dummies_test],  axis=1)

# ── Валидация family one-hot ──────────────────────────────────────────────────
family_cols = [c for c in df_train.columns if c.startswith("family_")]
n_families  = len(family_cols)

assert n_families == _EXPECTED_N_FAMILIES, (
    f"Ожидалось {_EXPECTED_N_FAMILIES} семейств, получено {n_families}"
)

family_row_sums = df_train[family_cols].sum(axis=1)
assert (family_row_sums == 1).all(), (
    f"Нарушение: {(family_row_sums != 1).sum()} строк не принадлежат ровно одному семейству"
)

train_fam_set = set(c for c in df_train.columns if c.startswith("family_"))
test_fam_set  = set(c for c in df_test.columns  if c.startswith("family_"))
assert train_fam_set == test_fam_set, (
    f"Несоответствие семейств train/test: {train_fam_set.symmetric_difference(test_fam_set)}"
)

print(f"family one-hot: {n_families} столбцов — ОК")
print(f"Каждая строка — ровно 1 семейство: ОК")
print(f"Признаковое пространство family train == test: ОК")

One-hot столбцов store_type: 5  — ОК
store_type столбцы: ['store_type_A', 'store_type_B', 'store_type_C', 'store_type_D', 'store_type_E']
cluster: [1, 17] — ОК
Каждая строка — ровно 1 тип магазина: ОК
Признаковое пространство train == test: ОК


## Ячейка 2.4.6 — Нормализация числовых признаков (StandardScaler)

In [18]:
from src.features.scaling import apply_standard_scaler

NUM_COLS_TO_SCALE = [
    "sales_weekly",
    "onpromotion_lag1",
    "rolling_mean_4",  "rolling_std_4",
    "rolling_mean_12",
] + [f"lag_{w}" for w in LAG_WEEKS]

missing_scale_cols = [c for c in NUM_COLS_TO_SCALE if c not in df_train.columns]
assert not missing_scale_cols, (
    f"NUM_COLS_TO_SCALE: столбцы отсутствуют в df_train: {missing_scale_cols}"
)

df_train_scaled, df_test_scaled, fitted_scaler = apply_standard_scaler(
    df_train, df_test, NUM_COLS_TO_SCALE
)

for col in NUM_COLS_TO_SCALE[:3]:
    col_mean = df_train_scaled[col].dropna().mean()
    col_std  = df_train_scaled[col].dropna().std()
    assert abs(col_mean) < 0.05, f"{col}: E[scaled_train]={col_mean:.4f}, ожидается ≈ 0"
    assert abs(col_std - 1.0) < 0.05, f"{col}: Std[scaled_train]={col_std:.4f}, ожидается ≈ 1"

for col in NUM_COLS_TO_SCALE[:3]:
    col_range = df_test_scaled[col].dropna().abs().max()
    assert col_range < 20, f"{col}: |max| тестовой выборки {col_range:.1f} — возможна аномалия"

print("\nСтатистика ПОСЛЕ нормализации (обучающая):")
for col in NUM_COLS_TO_SCALE[:4]:
    print(f"  {col:25s}: mean={df_train_scaled[col].mean():.4f}, std={df_train_scaled[col].std():.4f}")
print("\nfit-only-on-train: ОК")
print("Нормализованные датасеты сохранены в df_train_scaled / df_test_scaled")


Статистика ПОСЛЕ нормализации (обучающая):
  sales_weekly             : mean=0.0000, std=1.0000
  onpromotion_lag1         : mean=0.0000, std=1.0000
  rolling_mean_4           : mean=0.0000, std=1.0000
  rolling_std_4            : mean=-0.0000, std=1.0000

fit-only-on-train: ОК
Нормализованные датасеты сохранены в df_train_scaled / df_test_scaled


## Ячейка 2.4.7 — Итоговое признаковое пространство (Таблица 2.4 ПЗ)

In [19]:
_FEATURE_REGISTRY = [
    ("lag_1",            "лаг",           "train.csv / sales",          "Лаг 1 нед.: АКФ diff1, рис. 2.10 ПЗ"),
    ("lag_2",            "лаг",           "train.csv / sales",          "Лаг 2 нед.: АКФ diff1, рис. 2.10 ПЗ"),
    ("lag_4",            "лаг",           "train.csv / sales",          "Лаг 4 нед.: АКФ diff1, рис. 2.10 ПЗ"),
    ("lag_9",            "лаг",           "train.csv / sales",          "Лаг 9 нед.: ЧАКФ diff1=0,32, рис. 2.11 ПЗ"),
    ("lag_12",           "лаг",           "train.csv / sales",          "Лаг 12 нед.: АКФ diff1 лаги 12–13, рис. 2.10 ПЗ"),
    ("lag_52",           "лаг",           "train.csv / sales",          "Лаг 52 нед.: АКФ=0,306 > 0,1426 (табл. 2.3 ПЗ)"),
    ("onpromotion_lag1", "лаг",           "train.csv / onpromotion",   "Промо t-1: r=0,797 Пирсона, п. 2.2.1 ПЗ"),
    ("rolling_mean_4",  "скользящая",   "train.csv / sales",          "Локальный тренд 4 нед."),
    ("rolling_std_4",   "скользящая",   "train.csv / sales",          "Волатильность 4 нед. (шоки, п. 2.2.3 ПЗ)"),
    ("rolling_mean_12", "скользящая",   "train.csv / sales",          "Квартальный уровень 12 нед."),
    ("week_of_year_sin","циклический",  "date",                        "sin(2π·week/52); S=52 по АКФ=0,306, табл. 2.3 ПЗ"),
    ("week_of_year_cos","циклический",  "date",                        "cos(2π·week/52); непрерывность перехода 52→1"),
    ("month_sin",        "циклический",  "date",                        "sin(2π·month/12); месячная сезонность"),
    ("month_cos",        "циклический",  "date",                        "cos(2π·month/12); непрерывность перехода 12→1"),
    ("is_holiday",       "бинарный",     "holidays_events.csv",        "Объединение всех типов праздников"),
    ("is_national",      "бинарный",     "holidays_events.csv",        "Национальный: 174 записи, пики STL (п. 2.3 ПЗ)"),
    ("is_regional",      "бинарный",     "holidays_events.csv",        "Региональный: 24 события, локальные скачки"),
    ("store_type_A",     "one-hot",      "stores.csv / type",          "Тип A: 0,33–0,64 доли вариации (п. 2.2.1 ПЗ)"),
    ("store_type_B",     "one-hot",      "stores.csv / type",          "Тип B: аналогично store_type_A"),
    ("store_type_C",     "one-hot",      "stores.csv / type",          "Тип C: аналогично store_type_A"),
    ("store_type_D",     "one-hot",      "stores.csv / type",          "Тип D: аналогично store_type_A"),
    ("store_type_E",     "one-hot",      "stores.csv / type",          "Тип E: аналогично store_type_A"),
    ("cluster",          "числовой",     "stores.csv / cluster",       "Кластер 1–17; географическая однородность"),
]

feature_table = pd.DataFrame(
    _FEATURE_REGISTRY,
    columns=["признак", "тип", "источник", "экономический смысл"],
)

missing_features = [r for r in feature_table["признак"] if r not in df_train.columns]
if missing_features:
    print(f"[WARN] Признаки из реестра отсутствуют в df_train: {missing_features}")
else:
    print("Все признаки реестра присутствуют в df_train — ОК")

n_features = len(feature_table)
assert n_features == 23, f"Ожидалось 23 признака в реестре, получено {n_features}"
print(f"Итого признаков в реестре: {n_features}  (rolling_std_12 — вспомогательный, не входит)")

dup_features = feature_table["признак"][feature_table["признак"].duplicated()].tolist()
assert not dup_features, f"Дублирующиеся признаки в реестре: {dup_features}"

out_path = TABLES / "table_2_4_feature_space.csv"
feature_table.to_csv(out_path, index=False)
print(f"Таблица 2.4 сохранена: {out_path}")

display(feature_table)

Все признаки реестра присутствуют в df_train — ОК
Итого признаков в реестре: 23  (rolling_std_12 — вспомогательный, не входит)
Таблица 2.4 сохранена: D:\user\Documents\КПАиММО\store-sales-kr\reports\tables\table_2_4_feature_space.csv


,признак,тип,источник,экономический смысл
0,lag_1,лаг,train.csv / sales,"Лаг 1 нед.: АКФ diff1, рис. 2.10 ПЗ"
1,lag_2,лаг,train.csv / sales,"Лаг 2 нед.: АКФ diff1, рис. 2.10 ПЗ"
2,lag_4,лаг,train.csv / sales,"Лаг 4 нед.: АКФ diff1, рис. 2.10 ПЗ"
3,lag_9,лаг,train.csv / sales,"Лаг 9 нед.: ЧАКФ diff1=0,32, рис. 2.11 ПЗ"
4,lag_12,лаг,train.csv / sales,"Лаг 12 нед.: АКФ diff1 лаги 12–13, рис. 2.10 ПЗ"
5,lag_52,лаг,train.csv / sales,"Лаг 52 нед.: АКФ=0,306 > 0,1426 (табл. 2.3 ПЗ)"
6,onpromotion_lag1,лаг,train.csv / onpromotion,"Промо t-1: r=0,797 Пирсона, п. 2.2.1 ПЗ"
7,rolling_mean_4,скользящая,train.csv / sales,Локальный тренд 4 нед.
8,rolling_std_4,скользящая,train.csv / sales,"Волатильность 4 нед. (шоки, п. 2.2.3 ПЗ)"
9,rolling_mean_12,скользящая,train.csv / sales,Квартальный уровень 12 нед.


In [20]:
print("TRAIN_CUTOFF        :", TRAIN_CUTOFF)
print("df_train[date].max():", df_train[DATE_COL].max())
print("df_test[date].min() :", df_test[DATE_COL].min())

TRAIN_CUTOFF        : 2017-05-01
df_train[date].max(): 2017-04-24 00:00:00
df_test[date].min() : 2017-05-01 00:00:00


In [21]:
# ── Ячейка 2.4.8 — Перезапись parquet НЕДЕЛЬНЫМИ данными ──────────────────
# Ячейка 11 сохранила ДНЕВНЫЕ данные (2 872 584 строк, cols: sales_lag_1w...).
# Вследствие этого ноутбуки 03a/05/06 получают неверную схему и падают.
# Следовательно, перезаписываем features_*.parquet ЗДЕСЬ, после 2.4.x.

df_train.to_parquet(DATA_PROC / "features_train.parquet", index=False)
df_test.to_parquet(DATA_PROC  / "features_test.parquet",  index=False)

print("Артефакты перезаписаны недельными данными:")
print(f"  features_train.parquet : {len(df_train):>8,} строк | {df_train.shape[1]} стб.")
print(f"  features_test.parquet  : {len(df_test):>8,} строк | {df_test.shape[1]} стб.")
assert "lag_1" in df_train.columns, "FAIL: lag_1 отсутствует — проверить порядок ячеек"
assert df_train["lag_1"].dtype != object, "FAIL: lag_1 — строковый тип"
print("Схема parquet: ОК — lag_1 числовой, city/state отсутствуют (one-hot)")

Артефакты перезаписаны недельными данными:
  features_train.parquet :  400,950 строк | 30 стб.
  features_test.parquet  :   30,294 строк | 30 стб.
Схема parquet: ОК — lag_1 числовой, city/state отсутствуют (one-hot)


## Ячейка 13 — Финальная проверка качества признакового пространства

In [22]:
print("=" * 60)
print("ФИНАЛЬНАЯ ПРОВЕРКА ПОДРАЗДЕЛА 2.4")
print("=" * 60)

all_target_cols = (
    [f"lag_{w}" for w in LAG_WEEKS]
    + ["onpromotion_lag1"]
    + ["rolling_mean_4", "rolling_std_4", "rolling_mean_12"]
    + ["week_of_year_sin", "week_of_year_cos", "month_sin", "month_cos"]
    + ["is_holiday", "is_national", "is_regional"]
    + [f"store_type_{t}" for t in ["A","B","C","D","E"]]
    + ["cluster"]
)

for col in all_target_cols:
    assert col in df_train.columns, f"[FAIL] {col!r} отсутствует в df_train"
    assert col in df_test.columns,  f"[FAIL] {col!r} отсутствует в df_test"
print(f"[OK] Все {len(all_target_cols)} целевых признаков присутствуют в train и test")

non_lag_cols = [c for c in all_target_cols if not c.startswith("lag_")]
for col in non_lag_cols:
    n_nan = df_train[col].isna().sum()
    if n_nan > 0:
        print(f"[WARN] {col}: {n_nan} NaN в df_train")
print("[OK] Не-лаговые признаки: NaN-проверка завершена")

for col in ["is_holiday", "is_national", "is_regional"] + [f"store_type_{t}" for t in ["A","B","C","D","E"]]:
    assert df_train[col].isin([0, 1]).all(), f"[FAIL] {col}: содержит значения кроме 0/1"
print("[OK] Бинарные признаки: только значения 0 и 1")

assert df_train[DATE_COL].max() < pd.Timestamp(TRAIN_CUTOFF), "[FAIL] Утечка: train содержит test-период"
print("[OK] Утечка по времени: ОТСУТСТВУЕТ")

print(f"\ndf_train: {len(df_train):,} строк, {df_train.shape[1]} столбцов")
print(f"df_test : {len(df_test):,} строк, {df_test.shape[1]} столбцов")

print("=" * 60)
print("Подраздел 2.4 завершён. Запустите 02_stationarity_acf_pacf_adf.ipynb")
print("Для LSTM/ElasticNet используйте df_train_scaled / df_test_scaled")
print("Для XGBoost/RF используйте df_train / df_test (без нормализации)")

ФИНАЛЬНАЯ ПРОВЕРКА ПОДРАЗДЕЛА 2.4
[OK] Все 23 целевых признаков присутствуют в train и test
[WARN] onpromotion_lag1: 1782 NaN в df_train
[WARN] rolling_mean_4: 1782 NaN в df_train
[WARN] rolling_mean_12: 1782 NaN в df_train
[OK] Не-лаговые признаки: NaN-проверка завершена
[OK] Бинарные признаки: только значения 0 и 1
[OK] Утечка по времени: ОТСУТСТВУЕТ

df_train: 400,950 строк, 30 столбцов
df_test : 30,294 строк, 30 столбцов
Подраздел 2.4 завершён. Запустите 02_stationarity_acf_pacf_adf.ipynb
Для LSTM/ElasticNet используйте df_train_scaled / df_test_scaled
Для XGBoost/RF используйте df_train / df_test (без нормализации)
